In [5]:
import os
from typing import List, Dict, Any
import pandas as pd

In [6]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter

print("Loaded modules successfully.")

Loaded modules successfully.


## Document structure in Langchain


In [7]:
## Create a simple document

doc = Document(
    page_content="This is a sample document. It contains some text that we will use for testing the text splitting functionality.",
    metadata={"source": "sample_document.txt",
            "page": 1,
            "author": "BK",
            "date_created":"2026-00-17",
            "custom_field":"Any value"}
)

print("Document Structure")
print(f"Content: {doc.page_content}")
print(f"meta data : {doc.metadata}")

## why metadata is important
print("Metadata is crucial for:")
print("1. Source citation - shows where an answer came from (file, page, URL)")
print("2. Filtering - narrows search by author, date, category or document type")
print("3. Context after chunking - every chunk keeps its parent document's metadata")
print("4. Access control - hides documents from users who shouldn't see them")
print("5. Updates - dates and versions help replace old chunks with newer ones")
print("6. Debugging - shows which file and page a bad chunk came from")


Document Structure
Content: This is a sample document. It contains some text that we will use for testing the text splitting functionality.
meta data : {'source': 'sample_document.txt', 'page': 1, 'author': 'BK', 'date_created': '2026-00-17', 'custom_field': 'Any value'}
Metadata is crucial for:
1. Source citation - shows where an answer came from (file, page, URL)
2. Filtering - narrows search by author, date, category or document type
3. Context after chunking - every chunk keeps its parent document's metadata
4. Access control - hides documents from users who shouldn't see them
5. Updates - dates and versions help replace old chunks with newer ones
6. Debugging - shows which file and page a bad chunk came from


# Text Files (.txt) 

In [8]:
# Set root directory

# Notebook runs from notebooks/, so move up to the project root (safe to re-run)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"Current working directory: {os.getcwd()}")

file_path = os.path.join("data", "raw", "text", "deep_learning.txt")
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Characters read: {len(text)}")
print(text[:300])

Current working directory: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp
Characters read: 2505
Deep Learning: An Introduction

Deep learning is a subfield of machine learning that uses artificial neural networks with many layers to learn patterns from data. The word "deep" refers to the number of layers stacked between the input and the output. Each layer learns a more abstract representation


## Read single text file - TextLoader


In [9]:

from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/raw/text/machine_learning_algorithms.txt", encoding="utf-8")
docs = loader.load()

print(type(docs))
print(docs)

<class 'list'>
[Document(metadata={'source': 'data/raw/text/machine_learning_algorithms.txt'}, page_content="Machine Learning Algorithms: An Overview\n\nMachine learning is a branch of artificial intelligence in which computers learn patterns from data instead of following hand-written rules. A machine learning algorithm is the method a model uses to learn those patterns. Algorithms are usually grouped into three main types: supervised learning, unsupervised learning, and reinforcement learning.\n\nSupervised Learning\nIn supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised tasks are either regression (predicting a number) or classification (predicting a category).\n\nLinear Regression predicts a continuous value by fitting a straight line through the data. It is used for tasks such as predicting house prices or sales figures.\n\nLogistic Regression is used for c

/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_2194/3903396252.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [10]:
print(f"Loaded {len(docs)} documetns")
print(f"Content preview : {docs[0].page_content[:100]}..")
print(f"Metadata: {docs[0].metadata}")

Loaded 1 documetns
Content preview : Machine Learning Algorithms: An Overview

Machine learning is a branch of artificial intelligence in..
Metadata: {'source': 'data/raw/text/machine_learning_algorithms.txt'}


## DirectoryLoaders -  Multiple Text files

In [11]:
## Load all the text files from the directory

from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "data/raw/text",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True)

documents = dir_loader.load()

print(f"Loaded {len(documents)} documetns")

for i, doc in enumerate(documents):
    print(f"\nDocuments {i+1}")
    print(f" source : {doc.metadata['source']}")
    print(f" Length : {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 670.93it/s]

Loaded 2 documetns

Documents 1
 source : data/raw/text/machine_learning_algorithms.txt
 Length : 4220 characters

Documents 2
 source : data/raw/text/deep_learning.txt
 Length : 2505 characters


## DirectoryLoader: advantages and disadvantages

`DirectoryLoader` loads many files from a folder at once. It uses a pattern (`glob`) to choose files and passes each file to a loader class (`loader_cls`), such as `TextLoader`.

### ✅ Advantages

| Advantage | Description |
|---|---|
| **Loads files in bulk** | Loads every matching file with one call instead of one loader per file |
| **Pattern matching** | `glob="**/*.txt"` picks exactly which files to load, including those in subfolders |
| **Automatic metadata** | Each `Document` gets `source` set to its file path, which helps with citations and filtering |
| **Works with any loader** | `loader_cls` accepts `TextLoader`, `PyPDFLoader`, `CSVLoader` and others |
| **Progress bar** | `show_progress=True` shows progress for large folders |
| **Faster loading** | `use_multithreading=True` loads files in parallel |
| **Error handling** | `silent_errors=True` skips broken files instead of stopping the whole run |
| **Scales well** | New files dropped into the folder are picked up without code changes |

### ❌ Disadvantages

| Disadvantage | Description |
|---|---|
| **One loader class per call** | Every file uses the same `loader_cls`, so mixed folders (PDF + TXT + CSV) need separate `DirectoryLoader` calls |
| **Memory use** | All documents are loaded into memory at once, which is a problem for very large folders |
| **Encoding problems** | Files with different encodings can fail unless you pass `loader_kwargs={"encoding": "utf-8"}` or turn on `autodetect_encoding` |
| **Hidden failures** | `silent_errors=True` can hide files that failed to load |
| **Limited metadata** | Only `source` is added; author, date, category and similar fields must be added by hand |
| **No change tracking** | Every run reloads every file, even unchanged ones |
| **Extra dependencies** | Some setups need extra packages (for example `unstructured` for the default loader) |
| **Path mistakes** | Relative paths depend on the working directory, which is a common problem in notebooks |

### 💡 When to use it
- **Use `DirectoryLoader`** when a folder holds many files of the **same type**.
- **Use a single loader** such as `TextLoader` when you need just one file or custom handling for each file.


# Text Split Statergies

In [12]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter

print("Loaded modules successfully.")

Loaded modules successfully.


In [13]:
print(documents)

[Document(metadata={'source': 'data/raw/text/machine_learning_algorithms.txt'}, page_content="Machine Learning Algorithms: An Overview\n\nMachine learning is a branch of artificial intelligence in which computers learn patterns from data instead of following hand-written rules. A machine learning algorithm is the method a model uses to learn those patterns. Algorithms are usually grouped into three main types: supervised learning, unsupervised learning, and reinforcement learning.\n\nSupervised Learning\nIn supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised tasks are either regression (predicting a number) or classification (predicting a category).\n\nLinear Regression predicts a continuous value by fitting a straight line through the data. It is used for tasks such as predicting house prices or sales figures.\n\nLogistic Regression is used for classification. 

In [14]:
## Character Text splitter

text = documents[0].page_content
text

"Machine Learning Algorithms: An Overview\n\nMachine learning is a branch of artificial intelligence in which computers learn patterns from data instead of following hand-written rules. A machine learning algorithm is the method a model uses to learn those patterns. Algorithms are usually grouped into three main types: supervised learning, unsupervised learning, and reinforcement learning.\n\nSupervised Learning\nIn supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised tasks are either regression (predicting a number) or classification (predicting a category).\n\nLinear Regression predicts a continuous value by fitting a straight line through the data. It is used for tasks such as predicting house prices or sales figures.\n\nLogistic Regression is used for classification. It estimates the probability that an input belongs to a class, such as whether an email is sp

In [15]:
char_splitter = CharacterTextSplitter(
    separator="\n", # Split new line
    chunk_size=200, # Max chunk size
    chunk_overlap=20, #Overlap between chunks
    length_function=len # how to measuree chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Create {len(char_chunks)} chunks")
print(f"Fitst chunk : {char_chunks[0][:100]}")

Created a chunk of size 347, which is longer than the specified 200
Created a chunk of size 273, which is longer than the specified 200
Created a chunk of size 211, which is longer than the specified 200
Created a chunk of size 206, which is longer than the specified 200
Created a chunk of size 413, which is longer than the specified 200
Created a chunk of size 347, which is longer than the specified 200
Created a chunk of size 314, which is longer than the specified 200


Create 25 chunks
Fitst chunk : Machine Learning Algorithms: An Overview


In [16]:
print(char_chunks[0])
print("------------------------------------------------")
print(char_chunks[1])
print("------------------------------------------------")
print(char_chunks[2])
print("------------------------------------------------")
print(char_chunks[3])
print("------------------------------------------------")
print(char_chunks[4])

Machine Learning Algorithms: An Overview
------------------------------------------------
Machine learning is a branch of artificial intelligence in which computers learn patterns from data instead of following hand-written rules. A machine learning algorithm is the method a model uses to learn those patterns. Algorithms are usually grouped into three main types: supervised learning, unsupervised learning, and reinforcement learning.
------------------------------------------------
Supervised Learning
------------------------------------------------
In supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised tasks are either regression (predicting a number) or classification (predicting a category).
------------------------------------------------
Linear Regression predicts a continuous value by fitting a straight line through the data. It is used for tasks such as p

##  RecursiveCharacterTextSplitter

In [17]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], # Split new line
    chunk_size=200, # Max chunk size
    chunk_overlap=20, #Overlap between chunks
    length_function=len # how to measuree chunk size
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Create {len(recursive_chunks)} chunks")
print(f"Fitst chunk : {recursive_chunks[0][:100]}")

Create 34 chunks
Fitst chunk : Machine Learning Algorithms: An Overview


In [18]:
print(f" First Chunk : {recursive_chunks[0]}")

print(f" Seconf Chunk : {recursive_chunks[1]}")

print(f" Third Chunk : {recursive_chunks[2]}")

print(f" Fourth Chunk : {recursive_chunks[3]}")

print(f" fifth Chunk : {recursive_chunks[4]}")

 First Chunk : Machine Learning Algorithms: An Overview
 Seconf Chunk : Machine learning is a branch of artificial intelligence in which computers learn patterns from data instead of following hand-written rules. A machine learning algorithm is the method a model uses to
 Third Chunk : a model uses to learn those patterns. Algorithms are usually grouped into three main types: supervised learning, unsupervised learning, and reinforcement learning.
 Fourth Chunk : Supervised Learning
 fifth Chunk : In supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised tasks are either


In [19]:
text = "Machine learning models learn patterns from data using algorithms such as linear regression decision trees random forest and gradient boosting to make accurate predictions on new unseen examples. These models are trained on large datasets and evaluated using metrics like accuracy precision recall and F1 score to measure how well they generalize to real world data. Supervised learning uses labeled examples while unsupervised learning finds hidden structure in unlabeled data through clustering and dimensionality reduction techniques. Reinforcement learning trains an agent to take actions in an environment to maximize long term rewards through trial and error. Choosing the right algorithm depends on the size of the dataset the type of problem and the need for interpretability versus"


In [20]:
splitter = RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_text(text)
print(f"Created {len(chunks)} chunks")

for i in range(len(chunks) -1):
    print(f"chunk {i+1} '{chunks[i]}' ")
    print(f"chunk {i+2} '{chunks[i+1]}' ")

    print()

Created 5 chunks
chunk 1 'Machine learning models learn patterns from data using algorithms such as linear regression decision trees random forest and gradient boosting to make accurate predictions on new unseen examples.' 
chunk 2 'unseen examples. These models are trained on large datasets and evaluated using metrics like accuracy precision recall and F1 score to measure how well they generalize to real world data. Supervised' 

chunk 2 'unseen examples. These models are trained on large datasets and evaluated using metrics like accuracy precision recall and F1 score to measure how well they generalize to real world data. Supervised' 
chunk 3 'data. Supervised learning uses labeled examples while unsupervised learning finds hidden structure in unlabeled data through clustering and dimensionality reduction techniques. Reinforcement learning' 

chunk 3 'data. Supervised learning uses labeled examples while unsupervised learning finds hidden structure in unlabeled data through clusterin

In [25]:
## token based splitting
token_splitter = TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=20
)

token_chunks = token_splitter.split_text(text)

print(f"Created : {len(token_chunks)} chunks")
print(f"First chunk : {token_chunks[0]}")
print("-------------")
print(f"Second chunk : {token_chunks[2]}")


Created : 4 chunks
First chunk : Machine learning models learn patterns from data using algorithms such as linear regression decision trees random forest and gradient boosting to make accurate predictions on new unseen examples. These models are trained on large datasets and evaluated using metrics like accuracy precision recall and F1 score to measure
-------------
Second chunk :  Supervised learning uses labeled examples while unsupervised learning finds hidden structure in unlabeled data through clustering and dimensionality reduction techniques. Reinforcement learning trains an agent to take actions in an environment to maximize long term rewards through trial and error.


## CharacterTextSplitter vs RecursiveCharacterTextSplitter

### CharacterTextSplitter

Splits text using a **single separator** (default `"\n\n"`), and only falls back to raw character-count slicing if a chunk between separators still exceeds `chunk_size`.

**How it works:** looks for one separator string, splits on every occurrence, then merges the resulting pieces back together until each merged chunk is as close to `chunk_size` as possible (respecting `chunk_overlap`).

**Advantages**
- Simple and predictable — easy to reason about where splits happen
- Fast (single pass, one separator to search for)
- Good enough when your text has a very consistent structure (e.g., always double-newline-delimited paragraphs)

**Disadvantages**
- If the chosen separator doesn't appear often enough, it produces very uneven chunk sizes — or falls back to blind character slicing, which can cut mid-sentence or mid-word
- No fallback hierarchy — you're locked into one separator's granularity
- Poor at preserving semantic boundaries (sentences, paragraphs, code blocks) when the document structure is irregular

---

### RecursiveCharacterTextSplitter

Tries a **prioritized list of separators** (default `["\n\n", "\n", " ", ""]`), recursively splitting with the first separator, and only moving to the next (finer-grained) separator on chunks that are still too large.

**How it works:** attempt split on `"\n\n"` → any resulting piece still over `chunk_size` gets recursively split on `"\n"` → then `" "` → then individual characters as a last resort.

**Advantages**
- Keeps semantically related text together for longer (paragraphs > lines > words > characters), since it only breaks finer when it has to
- More robust across messy/inconsistent document structures — no need for uniform formatting
- This is why it's the **recommended default** in LangChain for general-purpose and RAG chunking
- Supports language-aware separator presets (`from_language(Language.PYTHON)`, etc.) for splitting code without breaking functions/classes

**Disadvantages**
- Slightly more computation (multiple passes/recursion vs. one)
- Still purely rule-based on character/whitespace patterns — not aware of actual sentence or semantic meaning (unlike embedding-based/semantic splitters)
- More separators to tune/understand than the simple version

---

### Quick comparison table

| | `CharacterTextSplitter` | `RecursiveCharacterTextSplitter` |
|---|---|---|
| Separators | One (default `"\n\n"`) | List, tried in order |
| Fallback on oversized chunk | Raw character slicing | Recurses to next, finer separator |
| Preserves semantic boundaries | Weak | Strong |
| Handles irregular formatting | Poorly | Well |
| Speed | Slightly faster | Slightly slower |
| Best for | Very uniformly structured text | General-purpose RAG ingestion (default choice) |

**Recommendation for a RAG pipeline:** use `RecursiveCharacterTextSplitter` unless your source documents have a rigidly consistent delimiter you can rely on — it produces more coherent chunks, which generally improves retrieval quality.
